# 🔍 AML Cycle Detection Engine
**System:** Anti-Money Laundering Detection | Neo4j Aura Enterprise 5.27  
**Module Coverage:** Module 3 → Module 7  
**Prerequisites:** Modules 1, 2, 2.5 must already be run (driver, eligible_seed_batches, eligible_seed_ids must be defined)

---

### Architecture
```
Module 3 : Cycle Detection Engine  (3-hop, 4-hop, 5-hop, 6-hop Cypher queries)
Module 4 : Merge & Remove Duplicates
Module 5 : Feature Engineering
Module 6 : Cycle Risk Scoring
Module 7 : Validation & Evaluation
```

---
## ⚙️ Setup
Mount Google Drive and install required packages.  
Skip this cell if your Drive is already mounted from the earlier modules.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install neo4j pandas scikit-learn --quiet

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.8 MB/s eta 0:00:00


---
## 🔌 Connect to Neo4j & Load Prerequisites
Re-establish the Neo4j driver connection and reload the eligible seed batches  
produced by Module 2.5. If you are running this notebook in the same Colab  
session as the earlier modules, the variables are already in memory — skip the CSV reload.

In [28]:
!pip install neo4j pandas scikit-learn --quiet

In [29]:
import pandas as pd
from google.colab import userdata
from neo4j import GraphDatabase

URI = userdata.get('NEO4J_URI').strip()
USERNAME = userdata.get('NEO4J_USERNAME').strip()
PASSWORD = userdata.get('NEO4J_PASSWORD').strip()
DATASET_DIR = "/content/drive/MyDrive/AML System/datasets"

TRANSACTIONS_CSV = f"{DATASET_DIR}/aml_sampled_dataset.csv"

BATCH_SIZE = 2000

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected to AuraDB")

Connected to AuraDB


### Selecting Candidate Accounts

In [30]:
# ============================================================
# MODULE 1 — CELL 1
# Compute transaction statistics for every account
# ============================================================

import pandas as pd
import numpy as np

print("Computing account statistics...")

ACCOUNT_STATS_QUERY = """
MATCH (a:Account)

OPTIONAL MATCH (a)-[out:TRANSACTION]->()

WITH
    a,
    count(out) AS out_tx_count,
    coalesce(sum(out.amount_paid), 0) AS total_out_amount

OPTIONAL MATCH ()-[inc:TRANSACTION]->(a)

WITH
    a,
    out_tx_count,
    total_out_amount,
    count(inc) AS in_tx_count,
    coalesce(sum(inc.amount_paid), 0) AS total_in_amount

RETURN

a.account_id AS account_id,
out_tx_count,
in_tx_count,
(out_tx_count + in_tx_count) AS transaction_count,
total_out_amount,
total_in_amount,
(total_out_amount + total_in_amount) AS total_transaction_amount
"""

with driver.session() as session:
    account_stats = session.run(ACCOUNT_STATS_QUERY).data()

account_df = pd.DataFrame(account_stats)

print(f"Accounts analysed : {len(account_df)}")

account_df.head()

Computing account statistics...
Accounts analysed : 43878


,account_id,out_tx_count,in_tx_count,transaction_count,total_out_amount,total_in_amount,total_transaction_amount
0,70_100428660,132870,850,133720,4.153677e+10,381704.99,4.153715e+10
1,1467_8013C4030,8,3,11,7.150404e+04,10300.00,8.180404e+04
2,119_811C597B0,26,39,65,4.017714e+07,66209821.35,1.063870e+08
3,70_100428858,7553,40,7593,2.276012e+09,19935.02,2.276032e+09
4,21174_800737690,20,8,28,5.924033e+06,55784.11,5.979818e+06


In [31]:
# Normalize features
def normalize(series):

    if series.max() == series.min():
        return pd.Series([0.5]*len(series), index=series.index)

    return (series-series.min())/(series.max()-series.min())

account_df["score_amount"] = normalize(
    account_df["total_transaction_amount"]
)

account_df["score_transactions"] = normalize(
    account_df["transaction_count"]
)

account_df.head()

,account_id,out_tx_count,in_tx_count,transaction_count,total_out_amount,total_in_amount,total_transaction_amount,score_amount,score_transactions
0,70_100428660,132870,850,133720,4.153677e+10,381704.99,4.153715e+10,1.596114e-01,1.000000
1,1467_8013C4030,8,3,11,7.150404e+04,10300.00,8.180404e+04,3.143417e-07,0.000075
2,119_811C597B0,26,39,65,4.017714e+07,66209821.35,1.063870e+08,4.088045e-04,0.000479
3,70_100428858,7553,40,7593,2.276012e+09,19935.02,2.276032e+09,8.745922e-03,0.056776
4,21174_800737690,20,8,28,5.924033e+06,55784.11,5.979818e+06,2.297816e-05,0.000202


In [32]:
# Compute Candidate Seed Score

account_df["candidate_seed_score"] = (

      0.60 * account_df["score_amount"]

    + 0.40 * account_df["score_transactions"]

).round(4)

print("Candidate seed scores computed successfully.")

account_df[[
    "account_id",
    "candidate_seed_score"
]].head()

Candidate seed scores computed successfully.


,account_id,candidate_seed_score
0,70_100428660,0.4958
1,1467_8013C4030,0.0000
2,119_811C597B0,0.0004
3,70_100428858,0.0280
4,21174_800737690,0.0001


In [33]:
# Select Top 10% Seed Accounts

account_df = account_df.sort_values(
    by=["candidate_seed_score", "account_id"],
    ascending=[False, True]
).reset_index(drop=True)

# Number of seed accounts (Top 10%)
num_seeds = int(len(account_df) * 0.10)

seed_accounts = account_df.head(num_seeds).copy()

# Create list of seed account IDs
seed_ids = seed_accounts["account_id"].tolist()

# Save for future modules
seed_accounts.to_csv(
    "/content/candidate_seed_accounts.csv",
    index=False
)

print("Candidate Seed Selection Completed")

print(f"Total Accounts          : {len(account_df)}")
print(f"Seed Accounts Selected  : {len(seed_ids)}")

seed_accounts.head(10)

Candidate Seed Selection Completed
Total Accounts          : 43878
Seed Accounts Selected  : 4387


,account_id,out_tx_count,in_tx_count,transaction_count,total_out_amount,total_in_amount,total_transaction_amount,score_amount,score_transactions,candidate_seed_score
0,70_100428780,12508,82,12590,2.602359e+11,3.345084e+06,2.602392e+11,1.000000,0.094145,0.6377
1,70_100428660,132870,850,133720,4.153677e+10,3.817050e+05,4.153715e+10,0.159611,1.000000,0.4958
2,70_100428738,10747,78,10825,1.711019e+11,5.298328e+06,1.711072e+11,0.657500,0.080946,0.4269
3,70_1004286A8,80389,512,80901,2.029276e+10,1.887023e+05,2.029294e+10,0.077978,0.605000,0.2888
4,70_1004287C8,9520,62,9582,6.513550e+10,2.679163e+06,6.513818e+10,0.250301,0.071650,0.1788
5,13029_805C2B8A0,0,1,1,0.000000e+00,6.641449e+10,6.641449e+10,0.255206,0.000000,0.1531
6,14381_805C2AFB0,1,1,2,6.641449e+10,1.643140e+03,6.641450e+10,0.255206,0.000007,0.1531
7,70_1004288E8,6291,38,6329,3.784097e+10,2.853579e+05,3.784126e+10,0.145409,0.047323,0.1062
8,213737_805DA8360,0,3,3,0.000000e+00,3.722570e+10,3.722570e+10,0.143044,0.000015,0.0858
9,70_1004286F0,11943,53,11996,1.440433e+10,1.242279e+05,1.440445e+10,0.055351,0.089703,0.0691


In [34]:
# ============================================================
# MODULE 2 — CELL 1
# Create Seed Batches
# ============================================================

from math import ceil

# Number of seed accounts to process together
BATCH_SIZE = 200

# Split seed_ids into batches
seed_batches = [
    seed_ids[i:i+BATCH_SIZE]
    for i in range(0, len(seed_ids), BATCH_SIZE)
]

print("="*60)
print("Seed Batch Creation Completed")
print("="*60)
print(f"Total Seed Accounts : {len(seed_ids)}")
print(f"Batch Size          : {BATCH_SIZE}")
print(f"Total Batches       : {len(seed_batches)}")
print("="*60)

Seed Batch Creation Completed
Total Seed Accounts : 4387
Batch Size          : 200
Total Batches       : 22


In [35]:
# ============================================================
# MODULE 2 — CELL 2
# Verify Seed Batches
# ============================================================

for i, batch in enumerate(seed_batches[:5]):
    print(f"Batch {i+1}")

    print(f"Number of Seeds : {len(batch)}")

    print(f"First 5 Seeds   : {batch[:5]}")

    print("-"*50)

Batch 1
Number of Seeds : 200
First 5 Seeds   : ['70_100428780', '70_100428660', '70_100428738', '70_1004286A8', '70_1004287C8']
--------------------------------------------------
Batch 2
Number of Seeds : 200
First 5 Seeds   : ['213737_805873FE0', '214609_80AEA8410', '215186_805869610', '218267_8118EDFA0', '21940_806A64F80']
--------------------------------------------------
Batch 3
Number of Seeds : 200
First 5 Seeds   : ['12_804B4EB10', '12_8050DEF60', '13078_804FE6A10', '136301_80FBF0B90', '13667_805FBC510']
--------------------------------------------------
Batch 4
Number of Seeds : 200
First 5 Seeds   : ['116_80EA45980', '117070_8064407B0', '117143_806545F80', '117143_80669D0F0', '117304_806B33320']
--------------------------------------------------
Batch 5
Number of Seeds : 200
First 5 Seeds   : ['10057_803AB8EB0', '10057_803D912B0', '10057_803D99600', '10057_803DE1580', '10057_803EBBCC0']
--------------------------------------------------


In [36]:
# ============================================================
# MODULE 2 — CELL 3
# Batch Summary
# ============================================================

batch_sizes = [len(batch) for batch in seed_batches]

print(f"Minimum Batch Size : {min(batch_sizes)}")

print(f"Maximum Batch Size : {max(batch_sizes)}")

print(f"Average Batch Size : {sum(batch_sizes)/len(batch_sizes):.2f}")

Minimum Batch Size : 187
Maximum Batch Size : 200
Average Batch Size : 199.41


In [37]:
# ============================================================
# MODULE 2.5 — CELL 1
# Filter Eligible Seed Accounts
# ============================================================

print("=" * 60)
print("Filtering Cycle Eligible Seed Accounts")
print("=" * 60)

ELIGIBLE_SEED_QUERY = """
UNWIND $seed_ids AS seed_id

MATCH (a:Account {account_id: seed_id})

OPTIONAL MATCH (a)-[:TRANSACTION]->()
WITH a, seed_id, count(*) AS out_degree

OPTIONAL MATCH ()-[:TRANSACTION]->(a)
WITH seed_id,
     out_degree,
     count(*) AS in_degree

WHERE out_degree > 0
  AND in_degree > 0

RETURN
    seed_id AS account_id,
    in_degree,
    out_degree,
    (in_degree + out_degree) AS total_degree

ORDER BY total_degree DESC
"""

with driver.session() as session:

    eligible_seeds = session.run(
        ELIGIBLE_SEED_QUERY,
        seed_ids=seed_ids
    ).data()

eligible_df = pd.DataFrame(eligible_seeds)

eligible_seed_ids = eligible_df["account_id"].tolist()

print(f"Original Seed Accounts : {len(seed_ids)}")
print(f"Eligible Seed Accounts : {len(eligible_seed_ids)}")
print(f"Filtered Out           : {len(seed_ids)-len(eligible_seed_ids)}")

Filtering Cycle Eligible Seed Accounts
Original Seed Accounts : 4387
Eligible Seed Accounts : 4387
Filtered Out           : 0


In [13]:
# ============================================================
# MODULE 2.5 — CELL 2
# Create Eligible Seed Batches
# ============================================================

BATCH_SIZE = 200

eligible_seed_batches = [

    eligible_seed_ids[i:i+BATCH_SIZE]

    for i in range(0, len(eligible_seed_ids), BATCH_SIZE)

]

print("=" * 60)
print("Eligible Seed Batch Summary")
print("=" * 60)
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Total Batches : {len(eligible_seed_batches)}")
print("=" * 60)

Eligible Seed Batch Summary
Batch Size    : 200
Total Batches : 22


In [14]:
# ============================================================
# MODULE 2.5 — CELL 3
# Verify Eligible Batches
# ============================================================

for i, batch in enumerate(eligible_seed_batches[:5]):

    print(f"Batch {i+1}")

    print(f"Accounts : {len(batch)}")

    print(batch[:5])

    print("-"*50)

Batch 1
Accounts : 200
['70_100428660', '70_1004286A8', '70_100428978', '70_100428780', '70_1004289C0']
--------------------------------------------------
Batch 2
Accounts : 200
['249176_81235EC40', '25981_803BE2A60', '116_80E8A63E0', '124331_80C09CDF0', '12979_80D2A4FE0']
--------------------------------------------------
Batch 3
Accounts : 200
['795_80242F5D0', '111312_8046BAD40', '12585_8104919A0', '223908_80C8A80D0', '3242_80BB4C4D0']
--------------------------------------------------
Batch 4
Accounts : 200
['17327_8051DD8F0', '17610_81084C820', '17615_8034B5CF0', '17907_807B27240', '17907_80B5D2DF0']
--------------------------------------------------
Batch 5
Accounts : 200
['19888_8078D0CB0', '19890_80CEF5970', '19890_80DFBED70', '19890_811C5CBC0', '19_809074D60']
--------------------------------------------------


In [15]:
import pandas as pd
import numpy as np
import time
from math import ceil
from google.colab import userdata
from neo4j import GraphDatabase

# ── Neo4j credentials from Colab Secrets ─────────────────────
URI      = userdata.get('NEO4J_URI').strip()
USERNAME = userdata.get('NEO4J_USERNAME').strip()
PASSWORD = userdata.get('NEO4J_PASSWORD').strip()

# ── Reconnect driver ──────────────────────────────────────────
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print('✓ Connected to Neo4j AuraDB')

# ── Rebuild eligible_seed_batches from eligible_seed_ids ──────
# eligible_seed_ids was produced by Module 2.5 above.
# We just re-slice it into batches here — no CSV reload needed.
BATCH_SIZE = 200
eligible_seed_batches = [
    eligible_seed_ids[i:i + BATCH_SIZE]
    for i in range(0, len(eligible_seed_ids), BATCH_SIZE)
]

print(f'✓ Eligible seed accounts : {len(eligible_seed_ids)}')
print(f'✓ Batches of {BATCH_SIZE}        : {len(eligible_seed_batches)}')


✓ Connected to Neo4j AuraDB
✓ Eligible seed accounts : 4387
✓ Batches of 200        : 22


---
## MODULE 3 — Cycle Detection Engine
---

### MODULE 3 — Cell 1 : Define the Four Fixed-Length Cycle Detectors
Each query uses a fully explicit hop-by-hop MATCH pattern (no `*` variable-length paths,  
no GDS, no APOC). Canonical ordering — enforcing that the origin must be the  
lexicographically smallest `account_id` in the cycle — eliminates duplicate results  
at the Cypher level so we don't over-count the same cycle from multiple seeds.

In [16]:
# ── 3-Hop Cycle : A → B → C → A ──────────────────────────────
# Simplest cycle form. Canonical guard: origin < every other node.
CYCLE_3_HOP = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})
  -[t1:TRANSACTION]->(b:Account)
  -[t2:TRANSACTION]->(c:Account)
  -[t3:TRANSACTION]->(a)

WHERE
    b.account_id <> c.account_id
AND b.account_id <> a.account_id
AND c.account_id <> a.account_id
AND a.account_id < b.account_id
AND a.account_id < c.account_id

RETURN
    a.account_id                                         AS origin_account,
    3                                                    AS cycle_length,
    [a.account_id, b.account_id, c.account_id,
     a.account_id]                                       AS account_chain,
    [t1.amount_paid, t2.amount_paid, t3.amount_paid]     AS amounts,
    [t1.timestamp,   t2.timestamp,   t3.timestamp]       AS timestamps,
    [t1.is_laundering, t2.is_laundering,
     t3.is_laundering]                                   AS is_laundering
"""

# ── 4-Hop Cycle : A → B → C → D → A ─────────────────────────
# 6 node-distinctness checks needed for 4 nodes (C(4,2) = 6).
CYCLE_4_HOP = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})
  -[t1:TRANSACTION]->(b:Account)
  -[t2:TRANSACTION]->(c:Account)
  -[t3:TRANSACTION]->(d:Account)
  -[t4:TRANSACTION]->(a)

WHERE
    b.account_id <> c.account_id
AND b.account_id <> d.account_id
AND c.account_id <> d.account_id
AND b.account_id <> a.account_id
AND c.account_id <> a.account_id
AND d.account_id <> a.account_id
AND a.account_id < b.account_id
AND a.account_id < c.account_id
AND a.account_id < d.account_id

RETURN
    a.account_id                                                     AS origin_account,
    4                                                                AS cycle_length,
    [a.account_id, b.account_id, c.account_id, d.account_id,
     a.account_id]                                                   AS account_chain,
    [t1.amount_paid, t2.amount_paid, t3.amount_paid,
     t4.amount_paid]                                                 AS amounts,
    [t1.timestamp,   t2.timestamp,   t3.timestamp,
     t4.timestamp]                                                   AS timestamps,
    [t1.is_laundering, t2.is_laundering, t3.is_laundering,
     t4.is_laundering]                                               AS is_laundering
"""

# ── 5-Hop Cycle : A → B → C → D → E → A ─────────────────────
# 10 node-distinctness checks needed for 5 nodes (C(5,2) = 10).
CYCLE_5_HOP = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})
  -[t1:TRANSACTION]->(b:Account)
  -[t2:TRANSACTION]->(c:Account)
  -[t3:TRANSACTION]->(d:Account)
  -[t4:TRANSACTION]->(e:Account)
  -[t5:TRANSACTION]->(a)

WHERE
    b.account_id <> c.account_id
AND b.account_id <> d.account_id
AND b.account_id <> e.account_id
AND c.account_id <> d.account_id
AND c.account_id <> e.account_id
AND d.account_id <> e.account_id
AND b.account_id <> a.account_id
AND c.account_id <> a.account_id
AND d.account_id <> a.account_id
AND e.account_id <> a.account_id
AND a.account_id < b.account_id
AND a.account_id < c.account_id
AND a.account_id < d.account_id
AND a.account_id < e.account_id

RETURN
    a.account_id                                                         AS origin_account,
    5                                                                    AS cycle_length,
    [a.account_id, b.account_id, c.account_id, d.account_id,
     e.account_id, a.account_id]                                         AS account_chain,
    [t1.amount_paid, t2.amount_paid, t3.amount_paid,
     t4.amount_paid, t5.amount_paid]                                     AS amounts,
    [t1.timestamp,   t2.timestamp,   t3.timestamp,
     t4.timestamp,   t5.timestamp]                                       AS timestamps,
    [t1.is_laundering, t2.is_laundering, t3.is_laundering,
     t4.is_laundering, t5.is_laundering]                                 AS is_laundering
"""

# ── 6-Hop Cycle : A → B → C → D → E → F → A ─────────────────
# Most expensive query (15 distinctness checks for 6 nodes).
# Run last; consider reducing batch size to 50 if Aura times out.
CYCLE_6_HOP = """
UNWIND $seed_ids AS origin_id

MATCH (a:Account {account_id: origin_id})
  -[t1:TRANSACTION]->(b:Account)
  -[t2:TRANSACTION]->(c:Account)
  -[t3:TRANSACTION]->(d:Account)
  -[t4:TRANSACTION]->(e:Account)
  -[t5:TRANSACTION]->(f:Account)
  -[t6:TRANSACTION]->(a)

WHERE
    b.account_id <> c.account_id
AND b.account_id <> d.account_id
AND b.account_id <> e.account_id
AND b.account_id <> f.account_id
AND c.account_id <> d.account_id
AND c.account_id <> e.account_id
AND c.account_id <> f.account_id
AND d.account_id <> e.account_id
AND d.account_id <> f.account_id
AND e.account_id <> f.account_id
AND b.account_id <> a.account_id
AND c.account_id <> a.account_id
AND d.account_id <> a.account_id
AND e.account_id <> a.account_id
AND f.account_id <> a.account_id
AND a.account_id < b.account_id
AND a.account_id < c.account_id
AND a.account_id < d.account_id
AND a.account_id < e.account_id
AND a.account_id < f.account_id

RETURN
    a.account_id                                                             AS origin_account,
    6                                                                        AS cycle_length,
    [a.account_id, b.account_id, c.account_id, d.account_id,
     e.account_id, f.account_id, a.account_id]                              AS account_chain,
    [t1.amount_paid, t2.amount_paid, t3.amount_paid,
     t4.amount_paid, t5.amount_paid, t6.amount_paid]                        AS amounts,
    [t1.timestamp,   t2.timestamp,   t3.timestamp,
     t4.timestamp,   t5.timestamp,   t6.timestamp]                          AS timestamps,
    [t1.is_laundering, t2.is_laundering, t3.is_laundering,
     t4.is_laundering, t5.is_laundering, t6.is_laundering]                  AS is_laundering
"""

# Bundle into an ordered dict — order matters: run cheaper detectors first
CYCLE_DETECTORS = {
    '3-hop': CYCLE_3_HOP,
    '4-hop': CYCLE_4_HOP,
    '5-hop': CYCLE_5_HOP,
    '6-hop': CYCLE_6_HOP,
}

print('Cycle detector queries defined:')
for name in CYCLE_DETECTORS:
    print(f'  ✓ {name}')

Cycle detector queries defined:
  ✓ 3-hop
  ✓ 4-hop
  ✓ 5-hop
  ✓ 6-hop


### MODULE 3 — Cell 2 : Single-Batch Executor Helper
Wraps one Cypher query execution against one batch of seed accounts.  
Error handling is isolated here so a single slow or timed-out batch  
does not crash the entire detection sweep.

In [17]:
def run_detector_on_batch(
    session,
    query: str,
    batch: list,
    detector_name: str
) -> list:
    """
    Execute one cycle-detector Cypher query for one seed batch.

    Parameters
    ----------
    session       : active Neo4j session
    query         : one of the CYCLE_*_HOP Cypher strings
    batch         : list of account_id strings for this batch
    detector_name : label used in warning messages

    Returns
    -------
    list of dicts — one dict per detected cycle (may be empty)
    """
    try:
        result = session.run(query, seed_ids=batch)
        # Materialise all records inside the session context window
        # so the cursor is not left open between batch iterations
        return result.data()

    except Exception as e:
        # Log the failure but continue — do not abort the full sweep
        print(f'  [WARN] {detector_name} | batch failed → {e}')
        return []


print('Single-batch executor helper defined ✓')

Single-batch executor helper defined ✓


### MODULE 3 — Cell 3 : Main Detection Loop
Outer loop iterates over all four detectors; inner loop iterates over every  
seed batch. Results are accumulated per detector in `raw_results`.  
Progress is printed every 10 batches so you can monitor long-running sweeps.

In [18]:
# Accumulator: one list per detector
raw_results = {name: [] for name in CYCLE_DETECTORS}

print('=' * 65)
print('Starting Cycle Detection Engine')
print(f'  Detectors : {list(CYCLE_DETECTORS.keys())}')
print(f'  Batches   : {len(eligible_seed_batches)}')
print(f'  Seeds     : {len(eligible_seed_ids)}')
print('=' * 65)

overall_start = time.time()

with driver.session() as session:

    for detector_name, query in CYCLE_DETECTORS.items():

        detector_start = time.time()
        detector_total = 0

        print(f'\n[{detector_name}] Running across {len(eligible_seed_batches)} batches...')

        for batch_idx, batch in enumerate(eligible_seed_batches):

            batch_results = run_detector_on_batch(
                session, query, batch, detector_name
            )

            # Accumulate this batch's findings
            raw_results[detector_name].extend(batch_results)
            detector_total += len(batch_results)

            # Print progress every 10 batches (avoid log spam)
            if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(eligible_seed_batches):
                elapsed = time.time() - detector_start
                print(
                    f'  Batch {batch_idx + 1:>4}/{len(eligible_seed_batches)} | '
                    f'Cycles so far: {detector_total:>6} | '
                    f'Elapsed: {elapsed:.1f}s'
                )

        print(f'[{detector_name}] Done — {detector_total} raw cycles')

total_elapsed = time.time() - overall_start
print(f'\nAll detectors finished in {total_elapsed:.1f}s')
print('\nRaw cycle counts:')
for name, records in raw_results.items():
    print(f'  {name} : {len(records)}')

Starting Cycle Detection Engine
  Detectors : ['3-hop', '4-hop', '5-hop', '6-hop']
  Batches   : 22
  Seeds     : 4387

[3-hop] Running across 22 batches...
  Batch   10/22 | Cycles so far:   1351 | Elapsed: 29.5s
  Batch   20/22 | Cycles so far:   1351 | Elapsed: 31.2s
  Batch   22/22 | Cycles so far:   1351 | Elapsed: 31.6s
[3-hop] Done — 1351 raw cycles

[4-hop] Running across 22 batches...
  Batch   10/22 | Cycles so far:   3560 | Elapsed: 4.9s
  Batch   20/22 | Cycles so far:   3560 | Elapsed: 7.7s
  Batch   22/22 | Cycles so far:   3560 | Elapsed: 8.0s
[4-hop] Done — 3560 raw cycles

[5-hop] Running across 22 batches...
  Batch   10/22 | Cycles so far:     82 | Elapsed: 3.5s
  Batch   20/22 | Cycles so far:     82 | Elapsed: 5.5s
  Batch   22/22 | Cycles so far:     82 | Elapsed: 5.8s
[5-hop] Done — 82 raw cycles

[6-hop] Running across 22 batches...
  Batch   10/22 | Cycles so far:     98 | Elapsed: 28.4s
  Batch   20/22 | Cycles so far:     98 | Elapsed: 45.5s
  Batch   22/22 |

---
## MODULE 4 — Merge & Remove Duplicates
---

### MODULE 4 — Cell 1 : Convert Raw Results to DataFrames & Merge
Each detector's list of dicts is converted to a DataFrame and tagged with  
a `detector` column for traceability. All four DataFrames are then  
concatenated into one combined DataFrame.

In [19]:
# Shared empty schema so concat never fails on empty detectors
EMPTY_SCHEMA = [
    'origin_account', 'cycle_length', 'account_chain',
    'amounts', 'timestamps', 'is_laundering', 'detector'
]

detector_dfs = {}

for detector_name, records in raw_results.items():

    if records:
        df = pd.DataFrame(records)
        df['detector'] = detector_name   # tag for traceability
        detector_dfs[detector_name] = df
        print(f'  {detector_name} : {len(df):>6} rows')
    else:
        # Empty DataFrame with correct columns — downstream code stays safe
        detector_dfs[detector_name] = pd.DataFrame(columns=EMPTY_SCHEMA)
        print(f'  {detector_name} :      0 rows (no cycles found)')

# Concatenate all detectors into one raw combined DataFrame
combined_raw_df = pd.concat(
    detector_dfs.values(),
    ignore_index=True
)

print(f'\nCombined raw DataFrame : {len(combined_raw_df)} rows')
combined_raw_df.head()

  3-hop :   1351 rows
  4-hop :   3560 rows
  5-hop :     82 rows
  6-hop :     98 rows

Combined raw DataFrame : 5091 rows


,origin_account,cycle_length,account_chain,amounts,timestamps,is_laundering,detector
0,121_8000E1590,3,"[121_8000E1590, 228101_80B0C3F40, 223_811B9B16...","[24760.86, 14373.69, 46043.6]","[2022-09-08T14:51:00.000000000+00:00, 2022-09-...","[1, 1, 1]",3-hop
1,121_8000E1590,3,"[121_8000E1590, 228101_80B0C3F40, 223_811B9B16...","[5582.98, 14373.69, 46043.6]","[2022-09-12T21:53:00.000000000+00:00, 2022-09-...","[1, 1, 1]",3-hop
2,121_8000E1590,3,"[121_8000E1590, 223_8119F8CC0, 222_811D80C30, ...","[51203.66, 56544.74, 51178.8]","[2022-09-08T16:19:00.000000000+00:00, 2022-09-...","[1, 1, 1]",3-hop
3,121_8000E1590,3,"[121_8000E1590, 223_811A65E30, 222_811D80C30, ...","[62690.22, 2023.38, 51178.8]","[2022-09-09T07:51:00.000000000+00:00, 2022-09-...","[1, 1, 1]",3-hop
4,121_8000E1590,3,"[121_8000E1590, 228101_80B0C3F40, 222_811D80C3...","[24760.86, 23526.06, 51178.8]","[2022-09-08T14:51:00.000000000+00:00, 2022-09-...","[1, 1, 1]",3-hop


### MODULE 4 — Cell 2 : Remove Duplicate Cycles
Although canonical ordering in Cypher eliminates most duplicates, the same cycle  
can appear twice if two seeds in different batches both qualify as the minimum-ID  
node. A sorted-node fingerprint catches these remaining cross-batch duplicates.

In [20]:
def make_cycle_fingerprint(account_chain: list) -> str:
    """
    Create an order-independent canonical fingerprint for a cycle.

    account_chain includes origin at both start and end:
      e.g. ['A', 'B', 'C', 'A']
    We drop the closing node, sort the rest, and join with '|'.
    This means ['A','C','B','A'] and ['A','B','C','A'] → 'A|B|C'
    """
    nodes = account_chain[:-1]        # strip the closing repeated origin
    return '|'.join(sorted(nodes))    # sort → join = canonical key


if not combined_raw_df.empty:

    # Apply fingerprint row-wise
    combined_raw_df['cycle_fingerprint'] = combined_raw_df['account_chain'].apply(
        make_cycle_fingerprint
    )

    rows_before = len(combined_raw_df)

    # Keep first occurrence — shorter cycles (3-hop) survive over longer ones
    # because detectors run in ascending hop order so 3-hop rows appear first
    combined_raw_df = combined_raw_df.drop_duplicates(
        subset=['cycle_fingerprint'],
        keep='first'
    ).reset_index(drop=True)

    rows_after = len(combined_raw_df)

    print(f'Rows before deduplication : {rows_before}')
    print(f'Rows after  deduplication : {rows_after}')
    print(f'Duplicates removed        : {rows_before - rows_after}')

else:
    combined_raw_df['cycle_fingerprint'] = pd.Series(dtype=str)
    print('No cycles detected — combined DataFrame is empty.')

Rows before deduplication : 5091
Rows after  deduplication : 169
Duplicates removed        : 4922


---
## MODULE 5 — Feature Engineering
---

### MODULE 5 — Cell 1 : Define Feature Engineering Function
Derives 12 analytical features from each cycle's amounts, timestamps, and  
laundering labels. These features feed the risk scorer in Module 6 and  
will also be used as node/edge features in the GNN training in Week 4–5.

In [21]:
from datetime import datetime, timezone


def to_epoch(ts) -> float | None:
    """Convert a Neo4j DateTime object or ISO string to a Unix timestamp."""
    if ts is None:
        return None
    if hasattr(ts, 'to_native'):           # neo4j DateTime object
        return ts.to_native().replace(tzinfo=timezone.utc).timestamp()
    if isinstance(ts, str):                # ISO string fallback
        return datetime.fromisoformat(ts).timestamp()
    return float(ts)                       # already numeric


def engineer_cycle_features(row: pd.Series) -> pd.Series:
    """
    Compute 12 derived features for a single detected cycle.

    Amount features  : total, min, max, avg, std, range
    Label features   : any_laundering, all_laundering, laundering_ratio
    Temporal features: time_span_seconds, avg_tx_interval, min_tx_interval
    """
    amounts    = row.get('amounts')       or []
    timestamps = row.get('timestamps')    or []
    labels     = row.get('is_laundering') or []

    # ── Amount features ───────────────────────────────────────
    amounts_clean = [a for a in amounts if a is not None]
    total_amount  = sum(amounts_clean)
    min_amount    = min(amounts_clean) if amounts_clean else 0.0
    max_amount    = max(amounts_clean) if amounts_clean else 0.0
    avg_amount    = float(np.mean(amounts_clean)) if amounts_clean else 0.0
    amount_std    = float(np.std(amounts_clean))  if len(amounts_clean) > 1 else 0.0
    amount_range  = max_amount - min_amount

    # ── Laundering label features ─────────────────────────────
    labels_clean     = [bool(l) for l in labels if l is not None]
    laundering_count = sum(labels_clean)
    n_labels         = len(labels_clean)
    any_laundering   = int(laundering_count > 0)
    all_laundering   = int(laundering_count == n_labels and n_labels > 0)
    laundering_ratio = (laundering_count / n_labels) if n_labels else 0.0

    # ── Temporal features ─────────────────────────────────────
    time_span_seconds = 0.0
    avg_tx_interval   = 0.0
    min_tx_interval   = 0.0

    try:
        epochs  = sorted([to_epoch(t) for t in timestamps if t is not None])
        epochs  = [e for e in epochs if e is not None]

        if len(epochs) >= 2:
            time_span_seconds = epochs[-1] - epochs[0]
            intervals         = [epochs[i+1] - epochs[i] for i in range(len(epochs)-1)]
            avg_tx_interval   = float(np.mean(intervals))
            min_tx_interval   = float(min(intervals))

    except Exception:
        # Leave temporal features as 0 if parsing fails
        pass

    return pd.Series({
        'total_amount'      : total_amount,
        'min_amount'        : min_amount,
        'max_amount'        : max_amount,
        'avg_amount'        : avg_amount,
        'amount_std'        : amount_std,
        'amount_range'      : amount_range,
        'any_laundering'    : any_laundering,
        'all_laundering'    : all_laundering,
        'laundering_ratio'  : laundering_ratio,
        'time_span_seconds' : time_span_seconds,
        'avg_tx_interval'   : avg_tx_interval,
        'min_tx_interval'   : min_tx_interval,
    })


print('Feature engineering function defined ✓')

Feature engineering function defined ✓


### MODULE 5 — Cell 2 : Apply Feature Engineering
Applies the function row-wise across all deduplicated cycles and joins  
the 12 new feature columns onto the base DataFrame to produce `enriched_df`.

In [22]:
if not combined_raw_df.empty:

    print('Applying feature engineering...')

    feature_df = combined_raw_df.apply(
        engineer_cycle_features,
        axis=1
    )

    # Horizontally join engineered features with base cycle columns
    enriched_df = pd.concat(
        [
            combined_raw_df.reset_index(drop=True),
            feature_df.reset_index(drop=True)
        ],
        axis=1
    )

    print(f'Feature engineering complete')
    print(f'  Cycles enriched : {len(enriched_df)}')
    print(f'  Total columns   : {len(enriched_df.columns)}')

    enriched_df[[
        'origin_account', 'cycle_length', 'total_amount',
        'laundering_ratio', 'time_span_seconds', 'any_laundering'
    ]].head()

else:
    enriched_df = combined_raw_df.copy()
    print('No cycles to engineer features for.')

Applying feature engineering...
Feature engineering complete
  Cycles enriched : 169
  Total columns   : 20


---
## MODULE 6 — Cycle Risk Scoring
---

### MODULE 6 — Cell 1 : Define Risk Scoring Functions
Each cycle receives a composite risk score (0.0–1.0) combining laundering labels,  
transaction amount, temporal velocity, and cycle hop length. Weights are  
calibrated toward AML signal strength — laundering ratio carries the most weight.

In [23]:
def compute_cycle_risk_score(row: pd.Series) -> float:
    """
    Composite risk score for one detected cycle. Range: 0.0 – 1.0

    Dimension              Weight   Rationale
    -------------------------------------------------------------
    Amount Score            0.35    Higher value cycles pose greater financial risk
    Velocity Score          0.30    Faster cycling indicates suspicious movement
    Hop Score               0.20    Longer cycles suggest sophisticated layering
    Amount Similarity       0.15    Similar transaction amounts indicate structured laundering

    NOTE:
    This score uses ONLY observable transaction behaviour.
    Ground-truth labels (is_laundering) are NOT used.
    """

    # ---------------------------------------------------------
    # 1. Amount Score
    # Higher total transaction amount = higher risk
    # ---------------------------------------------------------
    total = float(row.get('total_amount', 0.0))

    amount_score = (
        min(np.log10(total + 1) / 6.0, 1.0)
        if total > 0 else 0.0
    )

    # ---------------------------------------------------------
    # 2. Velocity Score
    # Faster cycles are more suspicious
    # ---------------------------------------------------------
    avg_interval = float(row.get('avg_tx_interval', 86400))

    velocity_score = (
        max(0.0, 1.0 - (avg_interval / 86400.0))
        if avg_interval > 0 else 1.0
    )

    # ---------------------------------------------------------
    # 3. Hop Score
    # Longer cycles are generally more sophisticated
    # ---------------------------------------------------------
    hop_length = int(row.get('cycle_length', 3))

    hop_score = (hop_length - 3) / 3.0

    # ---------------------------------------------------------
    # 4. Amount Similarity Score
    # Similar transaction amounts are suspicious
    # ---------------------------------------------------------
    std_amount = float(row.get("amount_std", 0.0))
    avg_amount = float(row.get("avg_amount", 1.0))

    similarity_score = max(
        0.0,
        1.0 - (std_amount / (avg_amount + 1))
    )

    # ---------------------------------------------------------
    # Final Weighted Risk Score
    # ---------------------------------------------------------
    score = (
          0.35 * amount_score
        + 0.30 * velocity_score
        + 0.20 * hop_score
        + 0.15 * similarity_score
    )

    return round(min(score, 1.0), 4)


def assign_risk_tier(score: float) -> str:
    """Map numeric risk score to investigation priority."""

    if score >= 0.75:
        return "CRITICAL"
    elif score >= 0.50:
        return "HIGH"
    elif score >= 0.25:
        return "MEDIUM"
    else:
        return "LOW"


print("Risk scoring functions defined ✓")

Risk scoring functions defined ✓


### MODULE 6 — Cell 2 : Apply Scoring → Produce Final `cycle_df`
Scores every enriched cycle and assigns a risk tier. Results are sorted  
descending by risk score — `cycle_df` is the canonical output of this  
detection module and the input to all downstream components.

In [24]:
if not enriched_df.empty:

    print('Computing risk scores...')

    enriched_df['risk_score'] = enriched_df.apply(
        compute_cycle_risk_score, axis=1
    )

    enriched_df['risk_tier'] = enriched_df['risk_score'].apply(
        assign_risk_tier
    )

    # Sort highest risk first; use cycle_length as tie-breaker (longer = more complex)
    cycle_df = enriched_df.sort_values(
        by=['risk_score', 'cycle_length'],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f'Risk scoring complete')
    print(f'  Total cycles scored : {len(cycle_df)}')
    print(f'\nRisk tier distribution:')
    print(cycle_df['risk_tier'].value_counts().to_string())

else:
    cycle_df = enriched_df.copy()
    cycle_df['risk_score'] = pd.Series(dtype=float)
    cycle_df['risk_tier']  = pd.Series(dtype=str)
    print('No cycles to score.')

cycle_df[[
    'origin_account', 'cycle_length', 'risk_score',
    'risk_tier', 'total_amount', 'laundering_ratio'
]].head(10)

Computing risk scores...
Risk scoring complete
  Total cycles scored : 169

Risk tier distribution:
risk_tier
HIGH        95
MEDIUM      73
CRITICAL     1


,origin_account,cycle_length,risk_score,risk_tier,total_amount,laundering_ratio
0,118_811B6E170,6,0.7522,CRITICAL,122561.65,1.0
1,118_811B6E170,6,0.6890,HIGH,182118.99,1.0
2,118_811B6E170,6,0.6656,HIGH,186862.30,1.0
3,118_811B6E170,5,0.6572,HIGH,80841.96,1.0
4,119_8000DB3C0,6,0.6544,HIGH,128273.00,1.0
5,118_811B6E170,6,0.6363,HIGH,305100.24,1.0
6,118_811B6E170,6,0.6303,HIGH,136809.11,1.0
7,118_811B6E170,6,0.6284,HIGH,220847.47,1.0
8,118_811B6E170,6,0.6224,HIGH,173059.85,1.0
9,118_811A67510,6,0.6203,HIGH,282815.92,1.0


---
## MODULE 7 — Validation & Evaluation
---

### MODULE 7 — Cell 1 : Classification Report & Confusion Matrix
Uses ground-truth `any_laundering` as the positive class and evaluates  
how well the risk scorer's threshold separates real laundering cycles  
from clean ones. Precision, recall, F1, and ROC-AUC are reported.

In [25]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print('=' * 65)
print('MODULE 7 — Validation & Evaluation')
print('=' * 65)

if cycle_df.empty:
    print('No cycles to evaluate.')

else:
    y_true = cycle_df['any_laundering'].astype(int)

    DETECTION_THRESHOLD = 0.25
    y_pred = (cycle_df['risk_score'] >= DETECTION_THRESHOLD).astype(int)

    print(f'\nDetection threshold : {DETECTION_THRESHOLD}')
    print(f'Total cycles        : {len(cycle_df)}')
    print(f'Ground-truth positives (any_laundering=1) : {y_true.sum()}')
    print(f'Predicted  positives (score >= threshold) : {y_pred.sum()}')

    # Dynamically build target_names based on classes actually present
    present_classes = sorted(y_true.unique())
    class_names     = {0: 'Clean', 1: 'Suspicious'}
    target_names    = [class_names[c] for c in present_classes]

    print('\nClassification Report:')
    print(classification_report(
        y_true, y_pred,
        labels=present_classes,
        target_names=target_names,
        zero_division=0
    ))

    print('Confusion Matrix (rows = actual, cols = predicted):')
    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    cm_df = pd.DataFrame(
        cm,
        index=  [f'Actual {n}'    for n in target_names],
        columns=[f'Pred {n}'      for n in target_names]
    )
    print(cm_df.to_string())

    # ROC-AUC only valid when both classes are present
    if y_true.nunique() > 1:
        auc = roc_auc_score(y_true, cycle_df['risk_score'])
        print(f'\nROC-AUC Score : {auc:.4f}')
    else:
        unique_class = class_names[y_true.iloc[0]]
        print(f'\nROC-AUC : skipped — all cycles are "{unique_class}" (only 1 class in ground truth)')
        print('This means no labeled laundering cycles were found in the detected set.')

MODULE 7 — Validation & Evaluation

Detection threshold : 0.25
Total cycles        : 169
Ground-truth positives (any_laundering=1) : 169
Predicted  positives (score >= threshold) : 169

Classification Report:
              precision    recall  f1-score   support

  Suspicious       1.00      1.00      1.00       169

    accuracy                           1.00       169
   macro avg       1.00      1.00      1.00       169
weighted avg       1.00      1.00      1.00       169

Confusion Matrix (rows = actual, cols = predicted):
                   Pred Suspicious
Actual Suspicious              169

ROC-AUC : skipped — all cycles are "Suspicious" (only 1 class in ground truth)
This means no labeled laundering cycles were found in the detected set.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


### MODULE 7 — Cell 2 : Per-Hop-Length Breakdown
Shows which detector (3/4/5/6-hop) catches the most laundering and  
where detection is weakest. Useful for tuning per-hop thresholds  
and for deciding which hop lengths to emphasise in GNN training.

In [26]:
if not cycle_df.empty:

    summary = cycle_df.groupby('cycle_length').agg(
        total_cycles      = ('risk_score',     'count'),
        avg_risk_score    = ('risk_score',     'mean'),
        laundering_cycles = ('any_laundering', 'sum'),
        avg_total_amount  = ('total_amount',   'mean'),
        critical_count    = ('risk_tier',      lambda x: (x == 'CRITICAL').sum()),
        high_count        = ('risk_tier',      lambda x: (x == 'HIGH').sum()),
    ).reset_index()

    summary['laundering_hit_rate'] = (
        summary['laundering_cycles'] / summary['total_cycles']
    ).round(4)

    print('Per-hop-length summary:')
    print(summary.to_string(index=False))

    print(f'\nOverall laundering hit rate : {cycle_df["any_laundering"].mean():.2%}')
    print(f'Mean risk score             : {cycle_df["risk_score"].mean():.4f}')
    print(f'CRITICAL tier cycles        : {(cycle_df["risk_tier"] == "CRITICAL").sum()}')

Per-hop-length summary:
 cycle_length  total_cycles  avg_risk_score  laundering_cycles  avg_total_amount  critical_count  high_count  laundering_hit_rate
            3            26        0.374619               26.0      5.691595e+06               0           1                  1.0
            4            33        0.440718               33.0      1.318673e+05               0           0                  1.0
            5            50        0.517004               50.0      1.644481e+05               0          35                  1.0
            6            60        0.595733               60.0      1.884267e+05               1          59                  1.0

Overall laundering hit rate : 100.00%
Mean risk score             : 0.5082
CRITICAL tier cycles        : 1


### MODULE 7 — Cell 3 : Save Final Outputs
Persists the full `cycle_df` and a top-50 high-risk subset to Google Drive.  
These files are consumed by the Layering, Structuring, and GNN modules  
in the weeks ahead, and by the investigator dashboard in Week 9–10.

In [27]:
# ============================================================
# Export all HIGH and CRITICAL risk cycles
# ============================================================

alerts_df = cycle_df[
    cycle_df["risk_tier"].isin(["HIGH", "CRITICAL"])
].copy()

# Sort by highest risk first
alerts_df = alerts_df.sort_values(
    by="risk_score",
    ascending=False
).reset_index(drop=True)

print("=" * 60)
print("HIGH & CRITICAL ALERTS")
print("=" * 60)
print(f"Total Alerts : {len(alerts_df)}")

print("\nRisk Tier Distribution:")
print(alerts_df["risk_tier"].value_counts())

alerts_df.head(10)

HIGH & CRITICAL ALERTS
Total Alerts : 96

Risk Tier Distribution:
risk_tier
HIGH        95
CRITICAL     1
Name: count, dtype: int64


,origin_account,cycle_length,account_chain,amounts,timestamps,is_laundering,detector,cycle_fingerprint,total_amount,min_amount,...,amount_std,amount_range,any_laundering,all_laundering,laundering_ratio,time_span_seconds,avg_tx_interval,min_tx_interval,risk_score,risk_tier
0,118_811B6E170,6,"[118_811B6E170, 118_812D06150, 223_8119F8CC0, ...","[19756.19, 19250.39, 8019.98, 20956.02, 19428....","[2022-09-10T11:13:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|118_812D06150|119_811DCA9B0|222_...,122561.65,8019.98,...,7893.224047,27130.24,1.0,1.0,1.0,196740.0,39348.0,6540.0,0.7522,CRITICAL
1,118_811B6E170,6,"[118_811B6E170, 222_800051110, 249118_814965B0...","[13199.87, 14131.83, 34929.54, 51178.8, 55819....","[2022-09-10T15:19:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|119_811DCA9B0|121_8000E1590|222_...,182118.99,12859.38,...,18104.271214,42960.19,1.0,1.0,1.0,256860.0,51372.0,7260.0,0.6890,HIGH
2,118_811B6E170,6,"[118_811B6E170, 222_800051110, 249118_814965B0...","[13199.87, 14131.83, 34929.54, 51178.8, 3220.9...","[2022-09-10T15:19:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|119_8000DB3C0|121_8000E1590|222_...,186862.30,3220.99,...,23566.551440,66980.28,1.0,1.0,1.0,256860.0,51372.0,11040.0,0.6656,HIGH
3,118_811B6E170,5,"[118_811B6E170, 118_812D06150, 223_8119F8CC0, ...","[19756.19, 19250.39, 8019.98, 20956.02, 12859.38]","[2022-09-10T11:13:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1]",5-hop,118_811B6E170|118_812D06150|119_811DCA9B0|223_...,80841.96,8019.98,...,4952.546144,12936.04,1.0,1.0,1.0,191760.0,47940.0,20520.0,0.6572,HIGH
4,119_8000DB3C0,6,"[119_8000DB3C0, 223_811A65E30, 121_8000E1590, ...","[15008.58, 15812.73, 24760.86, 20956.02, 19428...","[2022-09-12T11:04:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,119_8000DB3C0|119_811DCA9B0|121_8000E1590|222_...,128273.00,15008.58,...,5864.910365,17297.38,1.0,1.0,1.0,363540.0,72708.0,1800.0,0.6544,HIGH
5,118_811B6E170,6,"[118_811B6E170, 223_811B9B160, 121_8000E1590, ...","[48801.01, 46043.6, 51203.66, 56544.74, 32305....","[2022-09-17T00:03:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|119_8000DB3C0|121_8000E1590|222_...,305100.24,32305.96,...,11388.649687,37895.31,1.0,1.0,1.0,996120.0,199224.0,12360.0,0.6363,HIGH
6,118_811B6E170,6,"[118_811B6E170, 118_812D06150, 223_8119F8CC0, ...","[19756.19, 19250.39, 8019.98, 23526.06, 51178....","[2022-09-10T11:13:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|118_812D06150|121_8000E1590|222_...,136809.11,8019.98,...,13575.903973,43158.82,1.0,1.0,1.0,331200.0,66240.0,2400.0,0.6303,HIGH
7,118_811B6E170,6,"[118_811B6E170, 223_811B9B160, 121_8000E1590, ...","[48801.01, 46043.6, 30125.16, 25797.94, 34929....","[2022-09-17T00:03:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|118_812D06150|121_8000E1590|222_...,220847.47,25797.94,...,8178.092833,23003.07,1.0,1.0,1.0,966120.0,193224.0,18900.0,0.6284,HIGH
8,118_811B6E170,6,"[118_811B6E170, 118_812D06150, 223_8119F8CC0, ...","[19756.19, 19250.39, 8019.98, 23526.06, 32305....","[2022-09-10T11:13:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811B6E170|118_812D06150|119_8000DB3C0|222_...,173059.85,8019.98,...,19828.241157,62181.29,1.0,1.0,1.0,331200.0,66240.0,9060.0,0.6224,HIGH
9,118_811A67510,6,"[118_811A67510, 251264_812D1B280, 48308_811ED7...","[49167.87, 66719.61, 52436.18, 49642.18, 16723...","[2022-09-09T23:14:00.000000000+00:00, 2022-09-...","[1, 1, 1, 1, 1, 1]",6-hop,118_811A67510|119_81235E8B0|148389_811EDAA30|2...,282815.92,16723.65,...,14988.390544,49995.96,1.0,1.0,1.0,688680.0,137736.0,14880.0,0.6203,HIGH
